---
# `Agents Component of LangChain`
---

### Introduction
LLM: Providing us idea what tickets are the cheapest tot he Destination
AI Agents: Book the ticket

LLM vs AGent
- LLM provider information
- Agent: will get the work done for you because of Reasoning Capabilities and Tools

Eg. AI Agents: take today's Delhi Temperature 

Reasoning Capabilities: Chain of Thought Prompting, 

# `Detailed Notes`

# Agents in LangChain

> **Agent = An LLM-powered system that can decide what action to take, use tools, observe the result, and continue until it can produce a final answer.**

This is the key difference from a simple chain:

```text
Chain:
Input → Step A → Step B → Step C → Output

Agent:
Input → LLM decides → Tool → Observe → LLM decides → Tool → ... → Output
```

In current LangChain, agents combine **a language model + tools + instructions + state**, and the agent loop allows the model to dynamically decide which tool to call and when to stop. LangChain's `create_agent` is the standard high-level API for creating agents.

---

# 1. What is an Agent?

An **agent** is an application component that allows an LLM to:

1. Understand the user's request.
2. Decide what needs to be done.
3. Select an appropriate tool.
4. Call the tool.
5. Observe the tool's result.
6. Decide what to do next.
7. Repeat if necessary.
8. Return a final answer.

### Simple Example

User asks:

> "What is the weather in Delhi and should I carry an umbrella?"

The LLM cannot know the current weather from its training data.

An agent could reason about the task like:

```text
User Question
     ↓
Agent / LLM
     ↓
Need current weather
     ↓
Weather Tool
     ↓
Weather Result
     ↓
LLM
     ↓
Final Answer
```

---

# 2. Why Do We Need Agents?

A simple LLM can generate text:

```text
User → LLM → Answer
```

But real applications often need to **take actions**.

For example:

* Search the web
* Query a database
* Call an API
* Calculate something
* Read files
* Send an email
* Retrieve documents
* Execute application functions

An agent gives the LLM the ability to **choose among available tools**.

```text
                ┌→ Search
                │
User → Agent →  ├→ Database
                │
                ├→ Calculator
                │
                └→ API
```

---

# 3. Agent vs Chain

This is one of the **most important interview questions**.

## Chain

The workflow is predefined.

```text
Input
 ↓
Prompt
 ↓
LLM
 ↓
Parser
 ↓
Output
```

The developer decides the sequence.

---

## Agent

The next action can be selected dynamically.

```text
                ┌→ Tool A
                │
Input → Agent → ├→ Tool B
                │
                └→ Tool C
```

The LLM decides which tool is appropriate based on the task.

### Easy Memory Trick

> **Chain = Fixed workflow**

> **Agent = Dynamic workflow**

---

# 4. Agent Architecture

A simplified agent looks like:

```text
                  User
                   ↓
              Agent / LLM
                   ↓
            Decide next action
                   ↓
              ┌────┴────┐
              ↓         ↓
           Tool A     Tool B
              ↓         ↓
           Result      Result
              └────┬────┘
                   ↓
              Agent / LLM
                   ↓
             Final Answer
```

The important concept is the **loop**.

```text
Think/Decide
     ↓
Act
     ↓
Observe
     ↓
Decide Again
     ↓
Act Again
     ↓
...
     ↓
Final Answer
```

---

# 5. What is a Tool?

A **tool** is a function or capability that an agent can call to perform an action or retrieve information.

Examples:

```text
Calculator
Weather API
Web Search
Database Query
File Search
Email API
CRM API
Python Function
```

For example:

```python
def multiply(a: int, b: int) -> int:
    return a * b
```

This function can be exposed as a tool.

Then the agent can decide:

```text
User:
What is 25 × 40?

Agent:
Use calculator/multiply tool.

Tool:
1000

Agent:
The answer is 1000.
```

---

# 6. Tool Calling

Tool calling is central to agents.

The LLM doesn't necessarily execute the function itself.

Instead:

```text
LLM
 ↓
Tool Call Request
 ↓
Application executes tool
 ↓
Tool Result
 ↓
LLM
```

For example:

```text
LLM:
Call calculator with:
a = 25
b = 40

        ↓

Calculator:
1000

        ↓

LLM:
25 × 40 = 1000
```

This distinction is important:

> **The model decides to call the tool; the application/tool runtime executes the tool.**

---

# 7. Creating a Tool in LangChain

A simple tool can be defined using `@tool`.

```python
from langchain.tools import tool

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b
```

The function's:

* Name
* Arguments
* Type information
* Description

help the model understand when and how to use it.

---

# 8. Creating an Agent

A current LangChain approach uses `create_agent`.

Conceptually:

```python
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=[multiply],
    system_prompt="You are a helpful assistant."
)
```

Then:

```python
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is 25 multiplied by 40?"
        }
    ]
})
```

The agent can decide whether the tool is needed.

LangChain's current agent API is built around a model, tools, and system instructions, with the agent handling the tool-calling loop.

---

# 9. Agent Execution Flow

Suppose the user asks:

> "What is 125 × 48?"

The agent may follow:

```text
1. User Question
       ↓
2. Agent receives question
       ↓
3. LLM determines calculation is needed
       ↓
4. LLM requests calculator tool
       ↓
5. Tool executes
       ↓
6. Tool returns result
       ↓
7. LLM produces final answer
```

Conceptually:

```text
User
 ↓
Agent
 ↓
LLM
 ↓
Tool Call
 ↓
Calculator
 ↓
Result
 ↓
LLM
 ↓
Final Answer
```

---

# 10. Agent Loop

The agent loop is the heart of an agent.

```text
             ┌──────────────┐
             │     User     │
             └──────┬───────┘
                    ↓
             ┌──────────────┐
             │     Agent    │
             └──────┬───────┘
                    ↓
             ┌──────────────┐
             │      LLM     │
             └──────┬───────┘
                    ↓
             Decide Action
                    ↓
              ┌─────┴─────┐
              ↓           ↓
            Tool       Final Answer
              ↓
           Result
              ↓
             LLM
              ↓
        Decide Again
```

The loop continues until the model decides that it has enough information to answer.

---

# 11. Agent with Multiple Tools

Imagine we give an agent three tools:

```text
1. Calculator
2. Web Search
3. Database
```

User asks:

> "What was our company's revenue last year, and calculate the percentage increase from the previous year?"

The agent might decide:

```text
User Question
      ↓
Agent
      ↓
Need company revenue
      ↓
Database Tool
      ↓
Revenue Data
      ↓
Need calculation
      ↓
Calculator Tool
      ↓
Percentage Increase
      ↓
Final Answer
```

This is where agents become much more powerful than simple chains.

---

# 12. Agent vs LLM

Another important distinction.

### LLM

```text
Input → LLM → Text
```

The model generates a response.

### Agent

```text
Input
 ↓
LLM
 ↓
Decision
 ↓
Tool
 ↓
Observation
 ↓
LLM
 ↓
Decision
 ↓
Final Answer
```

Therefore:

> **An agent uses an LLM as its reasoning/decision-making component, but an agent is a larger system that also includes tools and execution logic.**

---

# 13. Agent vs Tool

These are also different.

### Tool

Performs a specific operation.

```text
Calculator
→ 25 × 40
```

### Agent

Decides **which tool to use and when**.

```text
User Question
     ↓
Agent
     ↓
Calculator?
Search?
Database?
     ↓
Decision
```

### Memory Trick

> **Tool = Can do**

> **Agent = Decides what to do**

---

# 14. Agent + Memory

Agents can also maintain conversation state.

```text
User
 ↓
Agent
 ↓
Memory / State
 ↓
LLM
 ↓
Tool
 ↓
Result
 ↓
Agent
 ↓
Response
```

For example:

```text
User:
My name is Arun.

Agent:
Nice to meet you.

User:
What is my name?

Agent:
Your name is Arun.
```

In modern LangChain, agent state can contain message history and other state, and persistence can be added using a checkpointer.

---

# 15. Agent + RAG

Agents can also use retrieval tools.

Suppose you have:

```text
Company Documents
       ↓
Vector Store
       ↓
Retriever
```

The retriever can be exposed to the agent as a tool.

Then:

```text
User
 ↓
Agent
 ↓
Need company information
 ↓
Retriever Tool
 ↓
Relevant Documents
 ↓
LLM
 ↓
Answer
```

So an agent can dynamically decide:

> "I need to search the company knowledge base."

---

# 16. Agent + RAG vs Traditional RAG

### Traditional RAG

The workflow is predefined:

```text
Question
 ↓
Retriever
 ↓
Context
 ↓
Prompt
 ↓
LLM
 ↓
Answer
```

### Agentic RAG

The agent decides whether and how to retrieve:

```text
Question
 ↓
Agent
 ↓
Need retrieval?
 ↓
Retriever Tool
 ↓
Documents
 ↓
Agent
 ↓
Answer
```

This can be useful when an application has multiple possible information sources.

---

# 17. Agent + Multiple Tools

A powerful GenAI application might have:

```text
                    Agent
                      │
        ┌─────────────┼─────────────┐
        ↓             ↓             ↓
    Web Search     Database      Calculator
        │             │             │
        ↓             ↓             ↓
     Results        Data          Result
        └─────────────┼─────────────┘
                      ↓
                     LLM
                      ↓
                  Final Answer
```

The agent chooses the appropriate tool based on the user's request.

---

# 18. When Should We Use an Agent?

Use an agent when:

* The next action is not known in advance.
* Multiple tools are available.
* The model needs to select tools dynamically.
* The task requires multiple steps.
* Different requests require different workflows.

### Example

```text
"Find the latest price of Bitcoin, compare it with yesterday's price, and calculate the percentage change."
```

Possible tools:

```text
Market API
Calculator
```

The agent can determine the sequence.

---

# 19. When Should We NOT Use an Agent?

This is equally important.

Don't use an agent simply because it sounds advanced.

If the workflow is predictable:

```text
Input
 ↓
Prompt
 ↓
LLM
 ↓
Parser
```

a simple chain is often better.

Agents can introduce:

* More latency
* More token usage
* More complexity
* More opportunities for incorrect tool selection
* More difficult debugging

### Rule

> **Use a chain when you know the workflow. Use an agent when the workflow needs dynamic decisions.**

---

# 20. Chain vs Agent vs LangGraph

This is an excellent interview comparison.

| Feature        | Chain                 | Agent            | LangGraph                       |
| -------------- | --------------------- | ---------------- | ------------------------------- |
| Workflow       | Predetermined         | Dynamic          | Graph-based                     |
| Tool selection | Usually predefined    | LLM can decide   | Can implement complex decisions |
| Branching      | Limited/simple        | Dynamic          | Strong                          |
| Loops          | Limited               | Agent loop       | Explicit graph loops            |
| State          | Basic                 | Supported        | Strong state management         |
| Complexity     | Low                   | Medium           | High                            |
| Best for       | Predictable pipelines | Tool-using tasks | Complex/stateful workflows      |

### Easy Memory Trick

```text
Chain
→ "I know the steps."

Agent
→ "The model decides the steps."

LangGraph
→ "I need a complex stateful workflow."
```

---

# 21. Agent Architecture in a Production Application

A realistic architecture might look like:

```text
                    User
                      ↓
                API / Backend
                      ↓
                    Agent
                      ↓
          ┌───────────┼───────────┐
          ↓           ↓           ↓
       Memory        RAG        Tools
          │           │           │
          ↓           ↓           ↓
       State       Vector DB    APIs/DB
          └───────────┼───────────┘
                      ↓
                     LLM
                      ↓
                Final Response
```

For example, an AI customer-support agent could have:

```text
Tools:
├── Search Knowledge Base
├── Get Order
├── Check Refund Status
├── Create Support Ticket
└── Send Email
```

The agent decides what is necessary.

---

# 22. Important Interview Questions

## Beginner

### Q1. What is an Agent in LangChain?

**Answer:**

An agent is an LLM-powered system that can dynamically decide which tools or actions to use to accomplish a task.

---

### Q2. What is the main difference between an agent and a chain?

**Answer:**

A chain follows a predefined sequence, while an agent can dynamically decide what action or tool to use next.

```text
Chain:
A → B → C

Agent:
A → Decide → B/C/D
```

---

### Q3. What is a tool?

**Answer:**

A tool is a callable function or capability that an agent can use to perform an action or retrieve information.

---

### Q4. What is tool calling?

**Answer:**

Tool calling is the process where the model requests execution of a specific tool with appropriate arguments, the application executes it, and the result is returned to the model.

---

# 23. Intermediate Interview Questions

### Q5. Why do agents need tools?

**Answer:**

LLMs are primarily language models. Tools allow an agent to access external information or perform actions such as calculations, database queries, API calls, searches, and application operations.

---

### Q6. Can an agent use multiple tools?

**Answer:**

Yes. An agent can be provided with multiple tools and can dynamically choose which tool or sequence of tools is appropriate.

---

### Q7. What is the agent loop?

**Answer:**

The agent loop is the repeated cycle of:

```text
Decide → Act → Observe → Decide Again
```

It continues until the agent determines that it can produce the final answer.

---

### Q8. Does the LLM execute the tool directly?

**Answer:**

Not necessarily. The model typically generates a structured tool-call request. The application/runtime executes the actual function and sends the tool result back to the model.

---

# 24. Scenario-Based Interview Questions

### Q9. You have a calculator, web search, and database tool. How would an agent use them?

**Answer:**

The agent receives the user's request and determines which tool is appropriate. It can call one or multiple tools, observe their results, and then generate the final response.

---

### Q10. When would you choose a chain instead of an agent?

**Answer:**

When the workflow is predictable and the steps are known in advance.

Example:

```text
Question
 ↓
Prompt
 ↓
LLM
 ↓
Parser
```

There is no need for dynamic decision-making.

---

### Q11. When would you choose an agent?

**Answer:**

When the application needs dynamic tool selection or the required sequence of actions cannot be determined beforehand.

---

### Q12. When would you choose LangGraph instead?

**Answer:**

For complex workflows involving state, branching, loops, persistence, human-in-the-loop steps, or more controlled orchestration.

---

# 25. Practical Example: AI Shopping Assistant

Imagine building an AI shopping assistant.

Available tools:

```text
search_products()
get_product_details()
check_inventory()
calculate_discount()
```

User asks:

> "Find me a laptop under ₹80,000 that is in stock and tell me the discounted price."

Agent workflow:

```text
                  User
                   ↓
                 Agent
                   ↓
          Search Products
                   ↓
             Candidates
                   ↓
          Check Inventory
                   ↓
             In-stock items
                   ↓
          Calculate Discount
                   ↓
             Final Price
                   ↓
              Agent/LLM
                   ↓
               Response
```

Notice that the agent can decide which tools are necessary.

---

# 26. Agent Safety

Agents can perform real actions, so safety matters.

For example:

```text
Agent
 ↓
Delete Database
```

could be dangerous.

Production agents should consider:

* Tool permissions
* Input validation
* Authentication
* Authorization
* Rate limits
* Human approval for sensitive actions
* Logging
* Timeouts
* Maximum iteration limits
* Error handling

### Important Principle

> **An agent should have only the tools and permissions it actually needs.**

---

# 27. Agent Observability

Agents can be harder to debug than simple chains because their execution path can vary.

For example:

```text
Request A:
Agent → Search → Calculator → Answer

Request B:
Agent → Database → Answer

Request C:
Agent → Search → Search Again → Answer
```

Therefore, tracing and evaluation are important.

Tools such as **LangSmith** can be used to trace, debug, evaluate, and monitor LangChain/LangGraph applications.

---

# 28. Key Takeaways

```text
Agent
↓
LLM-powered decision maker

Tool
↓
Callable capability/action

Tool Calling
↓
Model requests a tool execution

Agent Loop
↓
Decide → Act → Observe → Decide

Chain
↓
Fixed workflow

Agent
↓
Dynamic workflow

RAG
↓
Retrieve external knowledge

Memory
↓
Maintain state/information

LangGraph
↓
Complex stateful orchestration
```

---

# 29. 30-Second Revision

> **Agent = LLM + Tools + Dynamic Decision Making**

Remember:

```text
User
 ↓
Agent / LLM
 ↓
Decide
 ↓
Tool
 ↓
Observe Result
 ↓
Decide Again
 ↓
Final Answer
```

### Most Important Difference

```text
Chain
→ Fixed steps

Agent
→ Dynamic tool selection

LangGraph
→ Complex stateful workflows
```

### Tool

> **Tool performs the action.**

### Agent

> **Agent decides which action to take.**

---

# 30. 2-Minute Revision

## Agent

An agent is an LLM-powered system that dynamically decides what actions or tools are needed to solve a task.

## Core Components

```text
Agent
├── Model
├── Tools
├── Instructions
└── State
```

## Agent Loop

```text
User Input
    ↓
LLM
    ↓
Decision
    ↓
Tool Call
    ↓
Tool Result
    ↓
LLM
    ↓
Decision
    ↓
...
    ↓
Final Answer
```

## Chain vs Agent

```text
Chain:
A → B → C

Agent:
A → Decide → B/C/D
```

**Chain:** Use when the workflow is known.

**Agent:** Use when the model needs to dynamically decide what to do.

## Agent + RAG

```text
User
 ↓
Agent
 ↓
Retriever Tool
 ↓
Relevant Documents
 ↓
Agent/LLM
 ↓
Answer
```

## Agent + Memory

```text
User
 ↓
Agent
 ↓
Conversation State
 ↓
LLM
 ↓
Tools
 ↓
Response
```

## Final Interview Answer

> **An agent in LangChain is an LLM-powered workflow that can dynamically decide which tools to use and in what sequence to accomplish a task. Unlike a traditional chain, where the developer defines a fixed sequence of steps, an agent can follow a loop of deciding, calling tools, observing results, and deciding again until it can produce a final response. Modern LangChain provides `create_agent` as a high-level API for building agents, while more complex stateful orchestration can be handled with LangGraph.**
